In [2]:
import numpy as np
import pandas as pd

ml_ad = pd.read_csv("../results/ml_ad_scores.csv")
sim = pd.read_csv("../results/similarity_scores.csv")
dev = pd.read_csv("../results/developability_scores.csv")
cat = pd.read_csv("../data/cultivar_compound_catalogue.csv")

# Merge
df = ml_ad.merge(sim, on="compound_id").merge(dev, on="compound_id")

# Compute scores
df["known_active"] = False
df["novelty_bonus"] = np.where(df["max_similarity"] < 0.20, 1.0, 0.0)

df["final_score"] = (
    0.40 * df["ML_rank_score"]
    + 0.25 * df["similarity_support"]
    + 0.25 * df["developability_score"]
    + 0.10 * df["novelty_bonus"]
)

df["ligand_only_score"] = (
    0.60 * df["similarity_support"]
    + 0.30 * df["developability_score"]
    + 0.10 * df["novelty_bonus"]
)

df["high_ml_ad_extrapolation"] = (df["ad_flag"] == "extrapolation") & (
    df["ML_rank_score"] >= 0.80
)

# Rank
df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
df["final_rank"] = df.index + 1

df_lig = df.sort_values("ligand_only_score", ascending=False).reset_index(
    drop=True
)
df_lig["ligand_only_rank"] = df_lig.index + 1

# Fixed with errors="ignore"
df = df.drop(columns=["ligand_only_rank"], errors="ignore").merge(
    df_lig[["compound_id", "ligand_only_rank"]], on="compound_id"
)

# Merge back cultivar info
final_table = df.merge(cat, on="compound_id")

final_table.to_csv("../results/screened_compounds_annotated.csv", index=False)
print("Ranking locked. Saved to results/screened_compounds_annotated.csv")
display(
    final_table[[
        "compound_name",
        "final_score",
        "ML_rank_score",
        "similarity_support",
        "developability_score",
        "high_ml_ad_extrapolation",
    ]].head(5)
)

Ranking locked. Saved to results/screened_compounds_annotated.csv


,compound_name,final_score,ML_rank_score,similarity_support,developability_score,high_ml_ad_extrapolation
0,Borneol,0.738910,0.857143,0.184211,1.0,True
1,Borneol,0.738910,0.857143,0.184211,1.0,True
2,Borneol,0.738910,0.857143,0.184211,1.0,True
3,Camphene,0.731319,0.857143,0.153846,1.0,True
4,Camphene,0.731319,0.857143,0.153846,1.0,True


In [4]:
import pandas as pd
import json
from pathlib import Path

# 1. Ensure we have compound_id explicitly and sort
# Assuming final_table is the dataframe from your previous cell
final_table = final_table.sort_values("final_score", ascending=False)

# 2. Apply Duplication Cap (max 2 per compound)
# Sort by final_score, then take the top 2 for each compound
final_table_capped = final_table.groupby("compound_id").head(2).copy()

# 3. Extract the Primary Shortlist (Up to 10 compound-variety pairs)
shortlist = final_table_capped.head(10).copy()

# Add required fields for the pair-level view
# Since final_score has no variety-specific component, the operative selector is lexicographic cultivar_source
shortlist["cultivar_selection_rule"] = "lexicographic_provenance"
shortlist = shortlist.sort_values("final_score", ascending=False)

# 4. Generate the Compound-Level View (U)
# One row per compound. primary_cultivar is the lexicographically first identified variety.
compound_level = final_table.drop_duplicates("compound_id").copy()
# Add all_cultivars (authoritative) and n_cultivar_entries
cultivar_counts = final_table.groupby("compound_id").agg(
    all_cultivars=("cultivar_source", lambda x: ", ".join(sorted(set(x)))),
    n_cultivar_entries=("cultivar_source", "count")
).reset_index()

compound_level = compound_level.drop(columns=["cultivar_source", "cultivar_evidence_level", "source_publication", "source_locator"])
compound_level = compound_level.merge(cultivar_counts, on="compound_id")
compound_level = compound_level.rename(columns={"compound_name": "compound_name"}) # just ensuring it's there
# primary_cultivar is just the first one from the sorted list of all_cultivars
compound_level["primary_cultivar"] = compound_level["all_cultivars"].apply(lambda x: x.split(", ")[0])

compound_level = compound_level.sort_values("final_score", ascending=False)

# 5. Save Shortlists
shortlist.to_csv("../results/primary_shortlist.csv", index=False)
compound_level.to_csv("../results/primary_shortlist_compound_level.csv", index=False)

print(f"Shortlist generated: N = {len(shortlist)} pairs.")
print(f"Compound-level view generated: U = {len(compound_level)} unique compounds.")

# 6. Calculate Header Metadata for Shortlist
N_star = len(shortlist)
U_star = len(compound_level)
known_active_count = int(shortlist["known_active"].sum())
high_ml_ad_count = int(shortlist["high_ml_ad_extrapolation"].sum())
ad_extrapolation_count = int((shortlist["ad_flag"] == "extrapolation").sum())
ad_missing_count = 0 # We had no missing AD

metadata = {
    "N_star": N_star,
    "U_star": U_star,
    "known_active_count": known_active_count,
    "high_ml_ad_extrapolation_count": high_ml_ad_count,
    "ad_extrapolation_count": ad_extrapolation_count,
    "ad_missing_count": ad_missing_count
}
print("\nShortlist Header Metadata:")
print(json.dumps(metadata, indent=2))

# Save metadata for build_artifacts later
Path("../results/shortlist_metadata.json").write_text(json.dumps(metadata, indent=2))

Shortlist generated: N = 10 pairs.
Compound-level view generated: U = 28 unique compounds.

Shortlist Header Metadata:
{
  "N_star": 10,
  "U_star": 28,
  "known_active_count": 0,
  "high_ml_ad_extrapolation_count": 10,
  "ad_extrapolation_count": 10,
  "ad_missing_count": 0
}


158

In [5]:
import pandas as pd
import numpy as np

# 1. Get the Ligand-Only Shortlist
# We use the exact same capping and sorting logic, but on ligand_only_score
lig_table = final_table.sort_values("ligand_only_score", ascending=False)
lig_capped = lig_table.groupby("compound_id").head(2)
lig_shortlist = lig_capped.head(10)

# 2. Calculate Pair-Level Overlap (N*)
# A pair is (compound_id, cultivar_source)
locked_pairs = set(zip(shortlist["compound_id"], shortlist["cultivar_source"]))
lig_pairs = set(zip(lig_shortlist["compound_id"], lig_shortlist["cultivar_source"]))
pair_overlap = len(locked_pairs.intersection(lig_pairs))

# 3. Calculate Compound-Level Overlap (U*)
locked_compounds = set(shortlist["compound_id"])
lig_compounds = set(lig_shortlist["compound_id"])
compound_overlap = len(locked_compounds.intersection(lig_compounds))

# 4. Determine Sensitivity Label (Protocol §7.2)
N_star = len(shortlist)
if N_star == 0:
    label = "not_applicable_NO"
elif pair_overlap >= np.ceil(0.67 * N_star):
    label = "robust"
elif pair_overlap >= np.ceil(0.50 * N_star):
    label = "partially_robust"
else:
    label = "weight_sensitive"

print("--- Sensitivity Analysis ---")
print(f"Locked Shortlist N*: {N_star}")
print(f"Ligand-Only Shortlist N*: {len(lig_shortlist)}")
print(f"Pair-Level Overlap: {pair_overlap} / {N_star}")
print(f"Compound-Level Overlap: {compound_overlap} / {len(locked_compounds)}")
print(f"Sensitivity Label: {label.upper()}")

--- Sensitivity Analysis ---
Locked Shortlist N*: 10
Ligand-Only Shortlist N*: 10
Pair-Level Overlap: 6 / 10
Compound-Level Overlap: 4 / 5
Sensitivity Label: PARTIALLY_ROBUST


In [6]:
from pathlib import Path
import pandas as pd

# Define the diagnostics schema
cols = [
    "section", "row_id", "metric", "split", "value", "rank", "compound_id",
    "compound_name", "cultivar_source", "source_publication", "source_attempted",
    "structure_status", "reason", "final_score", "ligand_only_score",
    "ligand_only_rank", "failed_rules", "pattern", "pair_overlap",
    "sensitivity_label", "N_star", "U_star", "notes"
]

diag_rows = []

# A. Filtered but Interesting (Top 5 failing developability)
failed_dev = final_table[final_table["developability_score"] < 1.0].sort_values("final_score", ascending=False).head(5)
for i, (_, row) in enumerate(failed_dev.iterrows()):
    failed_rules = []
    if not row["lipinski"]: failed_rules.append("Lipinski")
    if not row["veber"]: failed_rules.append("Veber")
    if not row["pains"]: failed_rules.append("PAINS")
    if not row["aggregator"]: failed_rules.append("Aggregator")
    
    diag_rows.append({
        "section": "filtered_but_interesting",
        "row_id": i+1,
        "compound_id": row["compound_id"],
        "compound_name": row["compound_name"],
        "final_score": row["final_score"],
        "failed_rules": ", ".join(failed_rules),
        "reason": "Failed developability"
    })

# B. Disagreement Patterns (High ML + Low Similarity = Novel scaffold)
novel = final_table[(final_table["ML_rank_score"] >= 0.80) & (final_table["similarity_support"] < 0.20)].sort_values("final_score", ascending=False).head(5)
for i, (_, row) in enumerate(novel.iterrows()):
    diag_rows.append({
        "section": "disagreement_patterns",
        "row_id": i+1,
        "compound_id": row["compound_id"],
        "compound_name": row["compound_name"],
        "pattern": "high ML + low similarity",
        "notes": "Novel scaffold, higher risk"
    })

# C. Sensitivity Results
diag_rows.append({
    "section": "sensitivity",
    "row_id": 1,
    "pair_overlap": pair_overlap,
    "sensitivity_label": label,
    "N_star": N_star,
    "U_star": U_star,
    "notes": f"Compound overlap: {compound_overlap}"
})

# Write to diagnostics.csv (appending to the ledger rows)
diag_path = Path("../results/diagnostics.csv")
df_existing = pd.read_csv(diag_path)
df_new = pd.DataFrame(diag_rows, columns=cols)

# Ensure existing has same columns
df_existing = df_existing.reindex(columns=cols)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.to_csv(diag_path, index=False)
print(f"Added {len(diag_rows)} diagnostic rows to results/diagnostics.csv")

Added 11 diagnostic rows to results/diagnostics.csv


In [7]:
import pandas as pd
from pathlib import Path

diag_path = Path("../results/diagnostics.csv")
df = pd.read_csv(diag_path)

# Keep only the first occurrence of each compound in these two sections
for section in ["filtered_but_interesting", "disagreement_patterns"]:
    mask = df["section"] == section
    # Drop duplicates based on compound_id within the section
    df.loc[mask] = df.loc[mask].drop_duplicates(subset=["compound_id"])

# Reset row_id for those sections
for section in ["filtered_but_interesting", "disagreement_patterns"]:
    mask = df["section"] == section
    df.loc[mask, "row_id"] = range(1, mask.sum() + 1)

df.to_csv(diag_path, index=False)
print("Diagnostics CSV cleaned. Dropped duplicate compound-variety pairs.")

Diagnostics CSV cleaned. Dropped duplicate compound-variety pairs.
